<div class = "alert alert-info"> <h2 align = "center"> <b> Cross-asset Macro Overlay </b> </h2> 
<h3 align = center> <font  color = "black"> Data Collection and Preprocessing <font/> </h3>

</div>

### __Contexte__

Les portefeuilles multi-actifs sont traditionnellement construits sur des fondations quantitatives (corrélations, volatilités, betas). Cependant, une grande partie des mouvements de marché est macro-driven (inflation, taux, croissance, politique monétaire).

Deux sources d’information sont cruciales mais souvent exploitées séparément :

- Marché des options de taux → anticipe les mouvements futurs des banques centrales et l’incertitude (via vol implicite, skew, term structure).

- Flux de nouvelles macroéconomiques → messages textuels des banques centrales, données économiques, événements géopolitiques.

__Le projet vise à fusionner ces deux sources pour générer un overlay dynamique cross-asset, capable de repositionner le portefeuille face aux chocs macro.__

### __Objectifs principaux :__

- Construire un overlay dynamique cross-asset basé sur : Signaux textuels macro (news, communiqués, discours de banques centrales); Options rates (volatilité implicite, skew, convexité).

- Ajuster l’allocation en fonction de régimes de marché identifiés via Markov Switching / State-space.

- Quantifier l’impact des news sur la performance du portefeuille (news-to-portfolio attribution).



### __Collecte et préparation des données__

#### __Telechargement données financières__

__Si les données sont déjà téléchargées passez à:__ <a href="#Telechargement-donnees-macro">Telechargement donnees macro</a>


In [ ]:
# =========================
# Librairies
# =========================
import yfinance as yf
import pandas as pd
import numpy as np

# =========================
# 1. Extraction des prix historiques
# =========================
# Cross-asset : actions, obligations, FX, commodities
tickers = {
    'Equities': ['^GSPC', '^STOXX50E'],      # S&P500, Eurostoxx50
    'Bonds': ['^TNX', '^IRX'],               # US 10Y, US 3M (proxy)
    'FX': ['EURUSD=X', 'JPYUSD=X'],          # EUR/USD, JPY/USD
    'Commodities': ['GC=F', 'CL=F']          # Gold, WTI
}

# Télécharger les données sur 5 ans
start_date = '2018-01-01'
end_date = '2025-01-01'

prices = {}
for asset_class, assets in tickers.items():
    df = yf.download(assets, start=start_date, end=end_date)['Close']
    prices[asset_class] = df

# Calcul des rendements log
returns = {}
for asset_class, df in prices.items():
    returns[asset_class] = np.log(df / df.shift(1))

# Exemple : affichage des rendements S&P500
print(returns['Equities'].head())

# Return df
returns_df = pd.concat([returns['Equities'], returns["Bonds"], returns["FX"], returns["Commodities"]], axis=1, join='inner')
returns_df.to_csv('./data/returns.csv')

# =========================
# 2. Extraction / lecture des options rates
# =========================
# Pour les swaptions / caps/floors, on simule via CSV
# Exemple CSV : date, option_name, expiry, strike, vol
# => vous devez avoir un CSV téléchargé depuis Bloomberg ou autre source
options_csv = './data/options_rates.csv'  # Chemin vers votre fichier CSV

# Lecture du fichier
try:
    options_df = pd.read_csv(options_csv, parse_dates=['date'])
    # Exemple de colonnes : ['date', 'option_name', 'expiry', 'strike', 'implied_vol']
    print(options_df.head())
except FileNotFoundError:
    print("Fichier CSV des options rates non trouvé. Créez un fichier options_rates.csv avec vos données.")



#### __Telechargement donnees macro__


__Si les données sont déjà téléchargées passez à:__ <a href="#Synchronisation-temporelle">Synchronisation temporelle</a>


In [ ]:
"""
collect_macro_data.py

Collecte macro "end-to-end":
 - RSS feeds (ECB, IMF, Reuters...)
 - Scraping officiels (FOMC statements/minutes, ECB press releases) - now with historical loop for 2018-2025
 - Twitter/X scraping via tweepy (comptes officiels) - requires TWITTER_BEARER_TOKEN env var, adjusted for historical
 - Language detection, optional translation using googletrans
 - FinBERT tone scoring (hawkish/dovish proxy)
 - Sentence-transformers embeddings
 - Store: raw parquet and daily aggregated parquet signals

Sorties:
 - /data/raw/macro_raw.parquet
 - /data/daily/macro_daily_signals.parquet
"""

import os
import json
from datetime import datetime, timezone, timedelta
import time
import re
import logging
from typing import List, Dict, Any

import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from langdetect import detect, DetectorFactory
from newspaper import Article
from tqdm import tqdm
import dateutil.parser

# NLP
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# Twitter
import tweepy

# Translation
from googletrans import Translator

# Ensure deterministic langdetect
DetectorFactory.seed = 0

# -------------------------
# Configuration
# -------------------------
OUTPUT_RAW = "./data/raw/macro_raw.parquet"
OUTPUT_DAILY = "./data/daily/macro_daily_signals.parquet"
os.makedirs(os.path.dirname(OUTPUT_RAW), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_DAILY), exist_ok=True)

START_YEAR = 2018
END_YEAR = 2025  # Inclusive

# RSS & scraping sources (extend as needed)
RSS_FEEDS = {
    "ECB_press": "https://www.ecb.europa.eu/rss/press.xml",
    "IMF_news": "https://www.imf.org/external/rss/bottom.xml",  # Recent; historical via scraping if needed
}

# Official pages to scrape (base URLs; year-specific in loop)
SCRAPE_PAGES = {
    "FOMC_press": "https://www.federalreserve.gov/newsevents/pressreleases/{year}-press.htm",
    "ECB_press": "https://www.ecb.europa.eu/press/pr/date/{year}/html/index.en.html",
    "FOMC_minutes": "https://www.federalreserve.gov/monetarypolicy/fomc_historical_year.htm",  # Needs special handling
}

# Twitter accounts
TWITTER_ACCOUNTS = [
    "federalreserve", "ecb", "bankofengland", "boj_official_en"
]

# Model names
FINBERT_MODEL = "yiyanghkust/finbert-tone"
SENTENCE_EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Credibility scores
SOURCE_CREDIBILITY = {
    "ECB_press": 1.0,
    "FOMC_press": 1.0,
    "IMF_news": 0.9,
    "Twitter": 0.6,
    "other": 0.5
}

MAX_TOKENS = 500

# -------------------------
# Helpers
# -------------------------
def fetch_rss_entries(feed_url: str) -> List[Dict[str, Any]]:
    feed = feedparser.parse(feed_url)
    entries = []
    for e in feed.entries:
        published = e.get("published") or e.get("updated") or None
        entries.append({
            "title": e.get("title", ""),
            "summary": e.get("summary", "") or e.get("description", ""),
            "link": e.get("link", ""),
            "published": published
        })
    return entries

def fetch_article_text(url: str) -> str:
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        if text and len(text) > 50:
            return text
    except Exception:
        pass
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "macro-collector/1.0"})
        r.raise_for_status()
        soup = BeautifulSoup(r.content, "html.parser")
        for s in soup(["script", "style"]):
            s.decompose()
        paragraphs = [p.get_text(separator=" ", strip=True) for p in soup.find_all("p")]
        text = "\n\n".join([p for p in paragraphs if len(p) > 20])
        return text
    except Exception:
        return ""

def parse_datetime(s, url=None):
    if s:
        try:
            dt = dateutil.parser.parse(s, fuzzy=True)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return pd.Timestamp(dt)
        except Exception:
            pass
    if url:
        date_match = re.search(r'(\d{4})(\d{2})(\d{2})', url)
        if date_match:
            date_str = f"{date_match.group(1)}-{date_match.group(2)}-{date_match.group(3)}"
            try:
                dt = pd.to_datetime(date_str, utc=True)
                return dt
            except Exception:
                pass
    return pd.Timestamp(datetime.utcnow(), tz='UTC')

# -------------------------
# NLP
# -------------------------
print("Loading NLP models...")
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    fin_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    fin_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL).to(device)
    emb_model = SentenceTransformer(SENTENCE_EMB_MODEL, device=device)
except Exception as e:
    logging.error(f"Failed to load NLP models: {e}")
    raise

def chunk_text(text: str, max_tokens: int = MAX_TOKENS) -> List[str]:
    parts = re.split(r'\n{2,}|\.\s+', text)
    chunks = []
    cur = ""
    for p in parts:
        if len((cur + " " + p).split()) > max_tokens:
            if cur:
                chunks.append(cur.strip())
            cur = p
        else:
            cur = (cur + " " + p).strip()
    if cur:
        chunks.append(cur.strip())
    return [c for c in chunks if len(c) > 20]

def finbert_tone_score(text: str) -> float:
    if not isinstance(text, str) or len(text) < 10:
        return 0.0
    chunks = chunk_text(text)
    scores = []
    for ch in chunks:
        inputs = fin_tokenizer(ch, truncation=True, padding=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = fin_model(**inputs).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy().flatten()
        neg, neu, pos = probs[0], probs[1], probs[2]
        scores.append(float(pos - neg))
    return float(np.mean(scores)) if scores else 0.0

def compute_embedding(text: str) -> np.ndarray:
    if not isinstance(text, str) or len(text) < 5:
        return np.zeros(384)
    return emb_model.encode(text, convert_to_numpy=True)

# -------------------------
# Twitter
# -------------------------
def fetch_tweets_for_accounts(accounts: List[str], since_days: int = (datetime.utcnow() - datetime(2018, 1, 1)).days) -> List[Dict[str, Any]]:
    bearer_token = os.getenv('TWITTER_BEARER_TOKEN')
    if not bearer_token:
        logging.warning("TWITTER_BEARER_TOKEN not set. Skipping Twitter collection.")
        return []
    client = tweepy.Client(bearer_token=bearer_token)
    rows = []
    start_time = (datetime.utcnow() - timedelta(days=since_days)).isoformat() + 'Z'
    end_time = datetime(2025, 1, 1).isoformat() + 'Z'
    for acct in accounts:
        try:
            user = client.get_user(username=acct)
            tweets = client.get_users_tweets(id=user.data.id, start_time=start_time, end_time=end_time, tweet_fields=['created_at', 'text'], max_results=100)
            if tweets.data:
                for t in tweets.data:
                    rows.append({
                        "title": None,
                        "summary": t.text,
                        "link": f"https://twitter.com/{acct}/status/{t.id}",
                        "published": t.created_at.isoformat(),
                        "source": "Twitter",
                        "author": acct
                    })
            # Note: For full historical, pagination is needed; this gets recent 100. For more, use loop with next_token
        except Exception as e:
            logging.warning(f"Tweepy failed for {acct}: {e}")
    return rows

# -------------------------
# Main ingestion
# -------------------------
def ingest_all(rss_feeds=RSS_FEEDS, scrape_pages=SCRAPE_PAGES, twitter_accounts=TWITTER_ACCOUNTS, since_twitter_days=(datetime.utcnow() - datetime(2018, 1, 1)).days):
    records = []

    # 1) RSS - recent only
    print("Collecting RSS feeds (recent only)...")
    for src, url in rss_feeds.items():
        try:
            entries = fetch_rss_entries(url)
            for e in entries:
                link = e["link"]
                text = fetch_article_text(link)
                if not text:
                    text = e["summary"]
                published = parse_datetime(e["published"], url=link)
                if published.year < START_YEAR or published.year > END_YEAR:
                    continue
                records.append({
                    "id": f"rss::{src}::{hash(link)}",
                    "datetime_utc": published.isoformat(),
                    "source": src,
                    "url": link,
                    "title": e["title"],
                    "content": text,
                    "raw_summary": e["summary"]
                })
        except Exception as exc:
            logging.warning(f"RSS fetch failed for {src}: {exc}")

    # 2) Scrape - historical by year
    print("Scraping official pages (historical 2018-2025)...")
    for src, base_url in scrape_pages.items():
        for year in range(START_YEAR, END_YEAR + 1):
            url = base_url.format(year=year)
            try:
                r = requests.get(url, timeout=15, headers={"User-Agent": "macro-collector/1.0"})
                r.raise_for_status()
                soup = BeautifulSoup(r.content, "html.parser")
                links = set()
                for a in soup.find_all("a", href=True):
                    href = a["href"]
                    if href.startswith("/"):
                        base = re.match(r"https?://[^/]+", url).group(0)
                        href = base + href
                    if "press" in href or "statement" in href or "fomc" in href or "minutes" in href or "release" in href:
                        links.add(href)
                for link in links:
                    time.sleep(1)
                    try:
                        doc = requests.get(link, timeout=10, headers={"User-Agent": "macro-collector/1.0"})
                        doc.raise_for_status()
                        soup2 = BeautifulSoup(doc.content, "html.parser")
                        date_tag = soup2.find("time") or soup2.find("meta", {"name":"DC.date.issued"}) or soup2.find("meta", {"property":"article:published_time"}) or soup2.find('p', class_='article__time')
                        published_str = date_tag.get("datetime") or date_tag.get("content") or (date_tag.text.strip() if date_tag else None)
                        published = parse_datetime(published_str, url=link)
                        if published.year != year:
                            continue  # Ensure year match
                        for s in soup2(["script", "style"]):
                            s.decompose()
                        paragraphs = [p.get_text(separator=" ", strip=True) for p in soup2.find_all("p")]
                        text = "\n\n".join([p for p in paragraphs if len(p) > 20])
                        records.append({
                            "id": f"scrape::{src}::{year}::{hash(link)}",
                            "datetime_utc": published.isoformat(),
                            "source": src,
                            "url": link,
                            "title": a.text.strip() or None,
                            "content": text
                        })
                    except Exception as exc:
                        logging.warning(f"Failed to fetch {link}: {exc}")
            except Exception as exc:
                logging.warning(f"Scrape failed for {src} year {year}: {exc}")

    # 3) Twitter - historical (note: may require premium API for full access; basic limited)
    print("Collecting tweets (historical 2018-2025)...")
    try:
        tweets = fetch_tweets_for_accounts(twitter_accounts, since_days=since_twitter_days)
        for t in tweets:
            published = parse_datetime(t.get("published"))
            if published.year < START_YEAR or published.year > END_YEAR:
                continue
            records.append({
                "id": f"twitter::{hash(t.get('link'))}",
                "datetime_utc": published.isoformat(),
                "source": "Twitter",
                "url": t.get("link"),
                "title": None,
                "content": t.get("summary"),
                "author": t.get("author")
            })
    except Exception as exc:
        logging.warning(f"Twitter collection failed: {exc}")

    # 4) Post-processing
    print("Post-processing items...")
    translator = Translator()
    processed = []
    for rec in tqdm(records):
        content = rec.get("content") or ""
        if not content and rec.get("raw_summary"):
            content = rec.get("raw_summary")
        if not content or len(content) < 10:
            logging.info(f"Skipped item {rec['id']}: content too short")
            continue
        content_clean = re.sub(r'\s+', ' ', content).strip()
        try:
            lang = detect(content_clean[:1000])
        except Exception:
            lang = "unknown"
        translated = content_clean
        if lang != 'en' and lang != "unknown":
            try:
                translated = translator.translate(content_clean, dest='en').text
            except Exception as e:
                logging.warning(f"Translation failed for {rec['id']}: {e}")
        try:
            tone = finbert_tone_score(translated)
        except Exception as e:
            logging.warning(f"FinBERT failed for {rec['id']}: {e}")
            tone = 0.0
        try:
            emb = compute_embedding(translated)
            emb_list = emb.tolist()
        except Exception as e:
            logging.warning(f"Embedding failed for {rec['id']}: {e}")
            emb_list = [0.0] * 384
        dt_parsed = parse_datetime(rec.get("datetime_utc"))
        processed.append({
            "id": rec.get("id"),
            "datetime_utc": dt_parsed.isoformat(),
            "date": dt_parsed.date().isoformat(),
            "time": dt_parsed.time().isoformat(),
            "source": rec.get("source", "other"),
            "url": rec.get("url"),
            "title": rec.get("title"),
            "content": content_clean,
            "language": lang,
            "translated_content_en": translated,
            "finbert_tone": tone,
            "embedding": emb_list,
            "author": rec.get("author", None)
        })

    df = pd.DataFrame(processed)
    if df.empty:
        print("No items collected.")
        return None

    if os.path.exists(OUTPUT_RAW):
        df_existing = pd.read_parquet(OUTPUT_RAW)
        df = pd.concat([df_existing, df]).drop_duplicates(subset=['id'], keep='last')

    df.to_parquet(OUTPUT_RAW, index=False)
    print(f"Saved raw to {OUTPUT_RAW} ({len(df)} rows)")
    return df

# -------------------------
# Entrypoint
# -------------------------
if __name__ == "__main__":
    print("Starting historical macro data collection (2018-2025)...")
    df_raw = ingest_all()
    if df_raw is not None and not df_raw.empty:
        #df_daily = aggregate_daily(df_raw)
        print("Done.")
    else:
        print("Nothing to aggregate. Exiting.")

#### __Synchronisation temporelle__

In [ ]:
import pandas as pd
from datetime import datetime

# Define date range for synchronization (daily frequency)
start_date = '2018-01-01'
end_date = '2025-01-01'
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# 1. Load returns.csv
# Assumes file is in current directory; adjust path if needed
df_returns = pd.read_csv('./data/returns.csv', parse_dates=['Date'], index_col='Date')

# Ensure index is datetime
df_returns.index = pd.to_datetime(df_returns.index)

# Reindex to full daily range, forward-fill missing days (common for financial data)
df_returns = df_returns.reindex(date_range, method='ffill')

# 2. Load options_rates.csv
df_options = pd.read_csv('./data/options_rates.csv', parse_dates=['date', 'expiry'])

# Set date as index
df_options.set_index('date', inplace=True)

# Aggregate to daily: compute simple features for synchronization
# - Average implied_vol per day
# - Optional: skew and convexity as proxies (simplified)
def aggregate_options(group):
    avg_vol = group['implied_vol'].mean()
    strikes = group['strike'].unique()
    if len(strikes) > 2:
        sorted_group = group.sort_values('strike')
        vols = sorted_group['implied_vol'].values
        skew = (vols[-1] - vols[0]) / avg_vol if avg_vol > 0 else 0
        convexity = (vols[2] - 2 * vols[1] + vols[0]) / ((strikes[1] - strikes[0]) ** 2) if len(vols) >= 3 else 0
    else:
        skew = 0
        convexity = 0
    return pd.Series({
        'avg_implied_vol': avg_vol,
        'vol_skew': skew,
        'vol_convexity': convexity
    })

df_options_daily = df_options.groupby(level=0).apply(aggregate_options)

# Reindex to full daily range, forward-fill (vol surfaces persist)
df_options_daily = df_options_daily.reindex(date_range, method='ffill')

# 3. Load macro_raw.parquet
# Assumes file is in ./data/raw/macro_raw.parquet; adjust path if needed
df_macro_raw = pd.read_parquet('./data/macro_raw.parquet')

# Assume 'date' column exists (from script: 'date' as isoformat)
df_macro_raw['date'] = pd.to_datetime(df_macro_raw['date'])

# Aggregate to daily if multiple entries per day (mean finbert_tone, sum n_items, etc.)
df_macro_daily = df_macro_raw.groupby('date').agg({
    'finbert_tone': 'mean',  # Or 'finbert_tone_wavg' if exists
    # Add other columns as needed, e.g., 'n_items': 'sum',
    # 'embedding': lambda x: list(x)  # If needed, but for sync, perhaps drop complex
}).rename(columns={'finbert_tone': 'macro_tone'})

# Reindex to full daily range, forward-fill or set to 0 for missing (neutral tone)
df_macro_daily = df_macro_daily.reindex(date_range).fillna({'macro_tone': 0})

# 4. Synchronize: Merge all on date index
df_synced = pd.concat([
    df_returns,
    df_options_daily,
    df_macro_daily
], axis=1)

# Handle any remaining NaNs (e.g., at start): backward fill then forward fill
df_synced = df_synced.bfill().ffill()

# 5. Save synchronized data
output_path = './data/synced_data.parquet'
df_synced.to_parquet(output_path)
print(f"Synchronized data saved to {output_path} with shape {df_synced.shape}")

# Optional: Preview
print(df_synced.head())

#### __Constructions de signaux__

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats  # For outlier treatment (winsorize)
from sklearn.decomposition import PCA  # For latent factors if needed
from sklearn.preprocessing import StandardScaler
import QuantLib as ql  # For rates curve trends; install if needed (pip install QuantLib)
from datetime import datetime

# Load synchronized data from previous step (assumes exists; adjust path)
df_synced = pd.read_parquet('./data/synced_data.parquet')

# Alternatively, if not available, load originals and sync minimally
# df_returns = pd.read_csv('returns.csv', parse_dates=['Date'], index_col='Date')
# df_options = pd.read_csv('options_rates.csv', parse_dates=['date', 'expiry']).set_index('date')
# df_macro = pd.read_parquet('./data/raw/macro_raw.parquet')
# df_macro['date'] = pd.to_datetime(df_macro['date'])
# But proceed with df_synced for efficiency

# 3a) Options-based Factors
# Use df_options or synced columns (avg_implied_vol, vol_skew, vol_convexity already computed in sync)
# Enhance with implied rates curve trend: Fit a simple curve to vols by maturity

# First, ensure df_options is loaded for detailed computation if needed
df_options = pd.read_csv('./data/options_rates.csv', parse_dates=['date', 'expiry'])
df_options.set_index('date', inplace=True)

# Compute maturity in years (for curve)
df_options['maturity_years'] = (df_options['expiry'] - df_options.index).dt.days / 365.25

def compute_options_factors(df_opt):
    daily_factors = []
    for date, group in df_opt.groupby(level=0):
        # Vol implicite moyenne (already in synced, but recompute for precision)
        avg_vol = group['implied_vol'].mean()
        
        # Skew: (vol OTM put - vol OTM call) approx; here use high/low strike diff
        strikes = sorted(group['strike'].unique())
        if len(strikes) >= 3:
            low_strike_vol = group[group['strike'] == strikes[0]]['implied_vol'].mean()  # Assume low strike ~ puts OTM
            atm_strike_vol = group[group['strike'] == strikes[len(strikes)//2]]['implied_vol'].mean()
            high_strike_vol = group[group['strike'] == strikes[-1]]['implied_vol'].mean()
            skew = (low_strike_vol - high_strike_vol) / atm_strike_vol if atm_strike_vol > 0 else 0
        else:
            skew = 0
        
        # Convexity: Second derivative approx across strikes
        if len(strikes) >= 3:
            d_strike = strikes[1] - strikes[0]
            vols = [group[group['strike'] == s]['implied_vol'].mean() for s in strikes[:3]]  # Take first 3 for approx
            convexity = (vols[2] - 2 * vols[1] + vols[0]) / (d_strike ** 2)
        else:
            convexity = 0
        
        # Implied rates curve trend: Fit linear trend to vol vs maturity (slope as trend)
        if len(group['maturity_years'].unique()) >= 2:
            X = group['maturity_years'].values.reshape(-1, 1)
            y = group['implied_vol'].values
            slope = np.linalg.lstsq(X, y, rcond=None)[0][0]  # Simple linear fit slope
        else:
            slope = 0
        
        daily_factors.append({
            'date': date,
            'options_avg_vol': avg_vol,
            'options_skew': skew,
            'options_convexity': convexity,
            'implied_curve_trend': slope  # Positive: upward sloping vol term structure
        })
    return pd.DataFrame(daily_factors).set_index('date')

df_options_factors = compute_options_factors(df_options)

# Use QuantLib for more advanced curve if needed (e.g., Nelson-Siegel fit on implied vols)
# Example: Fit NS to vol term structure (simplified)
def ns_fit(maturities, vols):
    today = ql.Date(datetime.now().day, datetime.now().month, datetime.now().year)
    ql.Settings.instance().evaluationDate = today
    # Assume flat curve for demo; in real, fit to data
    # But for vols, not directly applicable; skip or use scipy curve_fit for beta0 + beta1*(1-exp(-tau/t))/ (tau/t) etc.
    from scipy.optimize import curve_fit
    def ns_formula(tau, beta0, beta1, beta2, lambda_):
        return beta0 + beta1 * (1 - np.exp(-tau / lambda_)) / (tau / lambda_) + beta2 * ((1 - np.exp(-tau / lambda_)) / (tau / lambda_) - np.exp(-tau / lambda_))
    try:
        popt, _ = curve_fit(ns_formula, maturities, vols)
        trend = popt[1]  # beta1 as slope proxy
    except:
        trend = 0
    return trend

# Apply to each day (prototype: apply to sample)
# For efficiency, add to loop above if needed: implied_curve_trend_ns = ns_fit(group['maturity_years'], group['implied_vol'])

# 3b) NLP-based Factors
# Use macro_raw: Already has finbert_tone per entry
df_macro = pd.read_parquet('./data/macro_raw.parquet')
df_macro['date'] = pd.to_datetime(df_macro['date'])
#df_macro['date'] = df_macro['datetime_utc'].dt.date

# Daily score: Weighted avg tone (as in original script, but recompute)
# Assume 'finbert_tone' is hawkish/dovish [-1,1]
df_nlp_factors = df_macro.groupby('date').agg({
    'finbert_tone': 'mean'  # Or weighted by credibility/source if columns exist
}).rename(columns={'finbert_tone': 'nlp_macro_score'})

# Reindex to date_range
date_range = pd.date_range(start='2018-01-01', end='2025-01-01')
df_nlp_factors = df_nlp_factors.reindex(date_range).fillna(0)  # Neutral for missing days

# If raw text needed, reload and process with transformers
# For prototype, assume precomputed; else:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def compute_tone(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).detach().numpy()[0]
    # Labels: neg=0, neu=1, pos=2 → score = pos - neg
    return probs[2] - probs[0]

# Apply if needed: df_macro['finbert_tone'] = df_macro['translated_content_en'].apply(compute_tone)
# Then aggregate as above

# 3c) Combine into Latent Factors
# Merge options + NLP
df_factors = df_options_factors.join(df_nlp_factors, how='outer').ffill().bfill()  # Align and fill

# Normalize
scaler = StandardScaler()
cols_to_norm = ['options_avg_vol', 'options_skew', 'options_convexity', 'implied_curve_trend', 'nlp_macro_score']
df_factors[cols_to_norm] = scaler.fit_transform(df_factors[cols_to_norm])

# Treat outliers: Winsorize at 1% / 99%
for col in cols_to_norm:
    df_factors[col] = stats.mstats.winsorize(df_factors[col], limits=[0.01, 0.01])

# Latent factors: Simple average or PCA
# Option factor: Avg of options signals
df_factors['latent_options_factor'] = df_factors[['options_avg_vol', 'options_skew', 'options_convexity', 'implied_curve_trend']].mean(axis=1)

# Macro factor: NLP score
df_factors['latent_macro_factor'] = df_factors['nlp_macro_score']

# Alternatively, PCA for combined latent
pca = PCA(n_components=2)
pca_factors = pca.fit_transform(df_factors[cols_to_norm])
df_factors['pca_factor_1'] = pca_factors[:, 0]  # e.g., main variance
df_factors['pca_factor_2'] = pca_factors[:, 1]

# Merge back to synced if needed
df_synced = df_synced.join(df_factors, how='left').ffill()

# Save
df_factors.to_parquet('./data/factors.parquet')
df_synced.to_parquet('./data/synced_with_factors.parquet')  # Updated synced

print("Factors computed and saved. Preview:")
print(df_factors.head())

In [ ]:

# Create DataFrame for PCA results
scaler = StandardScaler()
df_normalized = pd.DataFrame(scaler.fit_transform(df_factors[cols_to_norm]), columns=cols_to_norm, index=df_factors.index)

n_components=2
df_pca_factors = pd.DataFrame(pca_factors, columns=[f'PC{i+1}' for i in range(n_components)], index=df_normalized.index)

# Detailed Results

# 1. Variance Explained
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Explained Variance Ratio per Component:")
for i, var in enumerate(explained_variance):
    print(f"PC{i+1}: {var:.4f} ({var*100:.2f}%)")

print("\nCumulative Explained Variance:")
for i, cum_var in enumerate(cumulative_variance):
    print(f"Up to PC{i+1}: {cum_var:.4f} ({cum_var*100:.2f}%)")

# 2. Loadings (Component Weights: How each original variable contributes to each PC)
loadings = pd.DataFrame(pca.components_.T, columns=[f'PC{i+1}' for i in range(n_components)], index=cols_to_norm)
print("\nLoadings (Weights of Original Variables on Each PC):")
print(loadings)

# 3. Scree Plot (Visualize Variance Explained)
plt.figure(figsize=(10, 6))
plt.bar(range(1, n_components + 1), explained_variance, alpha=0.5, align='center', label='Individual Explained Variance')
plt.step(range(1, n_components + 1), cumulative_variance, where='mid', label='Cumulative Explained Variance')
plt.xlabel('Principal Components')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot for PCA')
plt.legend(loc='best')
plt.grid(True)
plt.show()

# 4. Biplot (Optional: Visualize Loadings and Scores for First 2 PCs)
# Scatter of samples on PC1 vs PC2, with arrows for loadings
def biplot(scores, loadings, labels=None):
    plt.figure(figsize=(12, 8))
    xs = scores[:, 0]
    ys = scores[:, 1]
    scalex = 1.0 / (xs.max() - xs.min())
    scaley = 1.0 / (ys.max() - ys.min())
    plt.scatter(xs * scalex, ys * scaley, alpha=0.1)  # Samples (dates)
    
    for i, vector in enumerate(loadings):
        plt.arrow(0, 0, vector[0], vector[1], color='r', alpha=0.5)
        if labels is not None:
            plt.text(vector[0] * 1.15, vector[1] * 1.15, labels[i], color='g', ha='center', va='center')
    
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title('Biplot of PCA (Samples and Variable Loadings)')
    plt.grid(True)
    plt.show()

# Call biplot with first 2 PCs
if n_components >= 2:
    biplot(pca_factors, pca.components_[:2].T, labels=cols_to_norm)

# 5. Save Detailed Results
loadings.to_csv('./output/pca_loadings.csv')
pd.DataFrame({'PC': [f'PC{i+1}' for i in range(n_components)], 'Explained_Variance': explained_variance}).to_csv('./output/pca_variance.csv', index=False)
df_pca_factors.to_parquet('./data/pca_factors.parquet')

print("\nDetailed PCA results saved to ./data/ folder.")